# Practice Lab: RAG with LangChain (Lesson 8.5)

Module 8 · 8 exercises · LCEL + EnsembleRetriever + LangGraph + agents + indexing

Exercises 1, 2, 4, 5, 7 run locally (pure Python simulations).


# Lesson 8.5: RAG with LangChain — Framework Power

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 3, 4, 5, 7 run **locally** (pure Python simulations of LangChain patterns).


In [ ]:
import numpy as np
import hashlib, math, re
from collections import Counter


---
## Exercise 1 (Easy): LCEL RAG Chain


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib
import numpy as np

class Runnable:
    def invoke(self, x): raise NotImplementedError
    def __or__(self, other): return RunnableSeq(self, other)
class RunnableSeq(Runnable):
    def __init__(self, a, b): self.a, self.b = a, b
    def invoke(self, x): return self.b.invoke(self.a.invoke(x))
class Fn(Runnable):
    def __init__(self, fn): self.fn = fn
    def invoke(self, x): return self.fn(x)
class Passthrough(Runnable):
    def invoke(self, x): return x
class Parallel(Runnable):
    def __init__(self, **kw): self.branches = kw
    def invoke(self, x): return {k: v.invoke(x) for k, v in self.branches.items()}

kb = {"refund": "Full refund 7 days. 50% 8-30 days.", "price": "14999 rupees lifetime.", "prereq": "Basic Python and math.", "live": "7 PM IST YouTube."}

def fake_embed(t, dim=64):
    h = hashlib.sha256(t.lower().encode()).digest()
    vec = np.array([b/255.0 for b in h[:dim]])
    for w, i in {"refund":0,"money":1,"price":2,"prereq":4,"python":5,"live":6}.items():
        if w in t.lower() and i < dim: vec[i] += 0.5
    return vec / np.linalg.norm(vec)

retriever = Fn(lambda q: max(kb.items(), key=lambda kv: np.dot(fake_embed(q), fake_embed(kv[1]))/(np.linalg.norm(fake_embed(q))*np.linalg.norm(fake_embed(kv[1]))))[1])
formatter = Fn(lambda ctx: f"Context: {ctx}")
prompter = Fn(lambda fmt: f"Answer from context ONLY.\n\n{fmt}\n\nAnswer:")
generator = Fn(lambda p: f"Based on records: {p.split('Context: ')[1].split(chr(10))[0]}")

chain = retriever | formatter | prompter | generator

print("LCEL RAG Chain:")
for q in ["How do I get money back?", "Prerequisites?", "Live class time?"]:
    print(f"  Q: {q}\n  A: {chain.invoke(q)[:60]}...\n")

par = Parallel(context=retriever, question=Passthrough())
r = par.invoke("refund policy")
print(f"Parallel: context='{r['context'][:30]}...', question='{r['question']}'")
print(f"\nPrimitives: Sequence(|), Parallel(dict), Passthrough, Lambda, Branch")


</details>


---
## Exercise 2 (Easy): EnsembleRetriever Hybrid Search


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import math, hashlib
import numpy as np
from collections import Counter

class BM25R:
    def __init__(self, docs):
        self.docs = docs; self.tok = [d.lower().split() for d in docs]
        self.avg = sum(len(d) for d in self.tok)/len(self.tok)
        self.df = Counter(); [self.df.update(set(d)) for d in self.tok]
        self.N = len(docs)
    def invoke(self, q, k=5):
        qt = q.lower().split(); scores = []
        for i, doc in enumerate(self.tok):
            s, tf = 0, Counter(doc)
            for w in qt:
                if w not in self.df: continue
                s += math.log((self.N-self.df[w]+0.5)/(self.df[w]+0.5)+1) * tf.get(w,0)*2.5/(tf.get(w,0)+1.5*(0.25+0.75*len(doc)/self.avg))
            scores.append((i, s))
        return sorted(scores, key=lambda x: -x[1])[:k]

class DenseR:
    def __init__(self, docs):
        self.docs = docs; self.embs = [self._e(d) for d in docs]
    def _e(self, t, dim=64):
        h = hashlib.sha256(t.lower().encode()).digest()
        v = np.array([b/255.0 for b in h[:dim]])
        for w, i in {"refund":0,"nts":2,"4021":3,"course":5,"prereq":7,"python":8,"live":9,"price":11}.items():
            if w in t.lower() and i < dim: v[i] += 0.5
        return v/np.linalg.norm(v)
    def invoke(self, q, k=5):
        qe = self._e(q)
        return sorted([(i, np.dot(qe, e)/(np.linalg.norm(qe)*np.linalg.norm(e))) for i, e in enumerate(self.embs)], key=lambda x: -x[1])[:k]

class EnsembleR:
    def __init__(self, retrievers, weights=None):
        self.rs = retrievers; self.ws = weights or [1.0]*len(retrievers)
    def invoke(self, q, k=3):
        scores = {}
        for r, w in zip(self.rs, self.ws):
            for rank, (idx, _) in enumerate(r.invoke(q)):
                scores[idx] = scores.get(idx, 0) + w/(60+rank+1)
        return sorted(scores.items(), key=lambda x: -x[1])[:k]

docs = ["Refund policy: full refund within 7 days", "Error NTS-4021: auth token expired",
        "GenAI course covers transformers RAG", "NTS-4021 troubleshooting: clear cache",
        "Prerequisites: basic Python math", "Course 14999 rupees lifetime access",
        "Live classes 7 PM IST YouTube recordings"]
bm25 = BM25R(docs); dense = DenseR(docs); ensemble = EnsembleR([bm25, dense])

for q, qt in [("get my money back","semantic"),("NTS-4021","exact"),("prerequisites price","mixed")]:
    br = bm25.invoke(q); dr = dense.invoke(q); er = ensemble.invoke(q)
    print(f"  Q: {q} ({qt})")
    print(f"    BM25:     [{br[0][0]}] {docs[br[0][0]][:40]}...")
    print(f"    Dense:    [{dr[0][0]}] {docs[dr[0][0]][:40]}...")
    print(f"    Ensemble: [{er[0][0]}] {docs[er[0][0]][:40]}...")

print(f"\nAdvanced: Ensemble, MultiQuery, ContextualCompression, ParentDoc, SelfQuery")
print(f"Impact: +26-31% NDCG over dense-only")


</details>


---
## Exercise 4 (Medium): LangGraph Self-Corrective RAG


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class StateGraph:
    def __init__(self):
        self.nodes, self.edges, self.cond_edges, self.entry = {}, {}, {}, None
    def add_node(self, name, fn): self.nodes[name] = fn
    def add_edge(self, a, b): self.edges[a] = b
    def add_conditional_edge(self, a, fn): self.cond_edges[a] = fn
    def set_entry_point(self, n): self.entry = n
    def invoke(self, state, max_steps=10):
        cur = self.entry; steps = 0
        while cur and cur != "END" and steps < max_steps:
            if cur in self.nodes: state = self.nodes[cur](state); state["trace"].append(cur)
            steps += 1
            if cur in self.cond_edges: cur = self.cond_edges[cur](state)
            elif cur in self.edges: cur = self.edges[cur]
            else: break
        return state

kb = {"refund": "Full refund 7 days. 50% 8-30.", "price": "14999 rupees lifetime.", "prereq": "Python and math.", "live": "7 PM IST YouTube."}

def retrieve(s):
    best = max(kb, key=lambda k: sum(1 for w in s["query"].lower().split() if w in k))
    s["docs"] = kb.get(best, "No info."); return s
def grade(s):
    qw = set(s["query"].lower().split()); dw = set(s["docs"].lower().split())
    s["rel"] = len(qw & dw)/max(len(qw),1); return s
def rewrite(s):
    s["retries"] = s.get("retries",0)+1
    for old, new in {"money":"refund","cost":"price","need":"prerequisite"}.items():
        if old in s["query"].lower(): s["query"] += f" {new}"
    return s
def generate(s):
    s["answer"] = f"Based on records: {s['docs'][:60]}"; return s
def route(s):
    if s["rel"] >= 0.2: return "generate"
    return "generate" if s.get("retries",0) >= 3 else "rewrite"

g = StateGraph()
g.add_node("retrieve", retrieve); g.add_node("grade", grade)
g.add_node("rewrite", rewrite); g.add_node("generate", generate)
g.set_entry_point("retrieve"); g.add_edge("retrieve","grade")
g.add_conditional_edge("grade", route); g.add_edge("rewrite","retrieve"); g.add_edge("generate","END")

print("LangGraph Self-Corrective RAG:")
for q in ["refund policy?", "how much money?", "GPT-5 news?"]:
    s = {"query": q, "docs":"", "rel":0, "answer":"", "retries":0, "trace":[]}
    r = g.invoke(s)
    print(f"  Q: {q} -> {' -> '.join(r['trace'])} (retries={r['retries']})")
    print(f"    A: {r['answer'][:50]}...")

print(f"\nLCEL: stateless linear. LangGraph: cycles, state, checkpoints.")


</details>


---
## Exercise 5 (Medium): Multi-Tool RAG Agent


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re

class Tool:
    def __init__(self, name, desc, fn): self.name, self.desc, self.fn = name, desc, fn
    def invoke(self, q): return self.fn(q)

def vs(q):
    kb = {"refund":"Full refund 7 days.","price":"14999 rupees.","prereq":"Python and math."}
    return next((f"[VS] {v}" for k, v in kb.items() if k in q.lower()), "[VS] No match.")
def ws(q): return f"[Web] Results for: {q[:25]}..."
def calc(q):
    nums = re.findall(r'[\d.]+', q)
    return f"[Calc] {float(nums[0])} * {float(nums[1])} = {float(nums[0])*float(nums[1]):.0f}" if len(nums) >= 2 else "[Calc] Need numbers."

tools = [Tool("vector_store","KB search",vs), Tool("web_search","Web search",ws), Tool("calculator","Math",calc)]

class Agent:
    def __init__(self, tools): self.tools = {t.name: t for t in tools}
    def _pick(self, q):
        q = q.lower()
        if any(k in q for k in ["calculate","multiply","total"]): return "calculator"
        if any(k in q for k in ["latest","news","current","today"]): return "web_search"
        return "vector_store"
    def invoke(self, q):
        t = self._pick(q); return {"tool": t, "result": self.tools[t].invoke(q)}

agent = Agent(tools)
for q in ["refund policy?", "latest GPT-5 news?", "calculate 14999 * 12", "prerequisites?"]:
    r = agent.invoke(q)
    print(f"  [{r['tool']:<14}] {q[:30]}... -> {r['result'][:40]}...")

print(f"\nPattern: @tool -> bind_tools() -> create_react_agent()")
print(f"AgentExecutor: DEPRECATED. Use LangGraph create_react_agent.")


</details>


---
## Exercise 7 (Challenge): Incremental Indexing Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib

class RecordManager:
    def __init__(self): self.records = {}
    def index(self, docs, cleanup=None):
        stats = {"added":0,"updated":0,"skipped":0,"deleted":0}
        ids = set()
        for d in docs:
            h = hashlib.md5(d["text"].encode()).hexdigest()[:12]
            ids.add(d["id"])
            if d["id"] in self.records:
                if self.records[d["id"]] == h: stats["skipped"] += 1
                else: self.records[d["id"]] = h; stats["updated"] += 1
            else: self.records[d["id"]] = h; stats["added"] += 1
        if cleanup == "full":
            for k in set(self.records) - ids: del self.records[k]; stats["deleted"] += 1
        elif cleanup == "incremental":
            pfx = docs[0]["id"].split("_")[0] if docs else ""
            for k in {k for k in self.records if k.startswith(pfx) and k not in ids}:
                del self.records[k]; stats["deleted"] += 1
        return stats

rm = RecordManager()
v1 = [{"id":f"faq_{i}","text":t} for i, t in enumerate(["Refund 7 days","14999 rupees","7 PM IST","Python math","EMI 2500"])]
print(f"Run 1 (initial): {rm.index(v1)}")
print(f"Run 2 (same):    {rm.index(v1)}")
v2 = [{"id":"faq_0","text":"Refund 14 days"},{"id":"faq_1","text":"19999 rupees"},
      {"id":"faq_2","text":"7 PM IST"},{"id":"faq_3","text":"Python math"}]
print(f"Run 3 (2 mod, 1 del, incremental): {rm.index(v2, cleanup='incremental')}")
print(f"Store: {len(rm.records)} records")
print(f"\nCleanup: None=dedup, incremental=stale source, full=all not in batch")


</details>


---
## Exercise 3 (Medium): Conversational RAG with History
See practice-lab-8_5.html for full code.


In [ ]:
# Exercise 3: Conversational RAG with History
pass


---
## Exercise 6 (Challenge): Production RAG with LangSmith
See practice-lab-8_5.html for full code.


In [ ]:
# Exercise 6: Production RAG with LangSmith
pass


---
## Exercise 8 (Challenge): LlamaIndex + LangChain Hybrid
See practice-lab-8_5.html for full code.


In [ ]:
# Exercise 8: LlamaIndex + LangChain Hybrid
pass
